# Generation of Black Hole Images: Thin Disk and Parametrized Models

This notebook generates black hole images for both **thin-disk** and **parametrized analytical models**.  
It works by computing geodesics for each pixel on the image plane and performing **forward integration of the intensity** along those geodesics.

Unlike the `ComputeGeodesics` notebook, this implementation is designed to be **memory efficient**:  
- It does **not** store all geodesic points in memory.  
- Instead, once the intensity for a pixel has been computed, the geodesic is discarded.  

As a result, this approach focuses directly on the **final intensity map**, avoiding the large memory overhead associated with retaining the full geodesic trajectories.


### Loading the Julia Files

Whenever you modify any of the source files, you need to reload `main.jl`.  
The file `main.jl` serves as a **wrapper** that imports and organizes all the functions of the code, ensuring that any changes are reflected in the notebook.


In [1]:
include("../src/main.jl")


main (generic function with 1 method)

### Analytic Parameters

In this cell we define the main parameters for the analytic model, including the observer position and screen setup, black hole spin, integration ranges, and observing frequency.  

These are the parameters used in the paper

**Remember:** 
- always update the black hole mass in `set_globals.jl`, not here.
- In `set_globals.jl` you can switch between "analytic" and "thin_disk"


In [2]:
#Analytic parameters

#Setting up the parameters
#Observer distance in Rg
ro = 1000.0
#Observer inclination in degrees
th = 60.0

#Observer azimuth in degrees
phi = 0.0

# Size of the screen in Rg in both directions
DXsize = 30.0
DYsize = 30.0

# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
fovx = DXsize/ro
fovy = DYsize/ro

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
res = 40
pixels_x = 40
pixels_y = 40
# Distance to the source in parsecs
SourceD = 7.778e3 * PC
bhspin = 0.9
Rout = 1000.0
Rstop = 10000.0
Rh = 1 + sqrt(1. - bhspin * bhspin);

cstartx = [0.0, log(Rh), 0.0, 0.0]
cstopx = [0.0, log(1000.0), 1.0, 2.0 * π]
# Frequency observed by the camera in Hz
freq = 230e9


2.3e11

These are the parameters used in the paper for the thin disk model

If you want to use "thin_disk" model, ***remember to change it in in `set_globals.jl`***


In [ ]:
#THIN DISK PARAMETERS


## REMEMBER TO CHANGE BLACK HOLE MASS IN PARAMETERS.JL

#Setting up the parameters
#Observer distance in Rg
ro = 10000.0
#Observer inclination in degrees
th = 60.0

#Observer azimuth in degrees
phi = 0.0

# Size of the screen in Rg in both directions
DXsize = 40.0
DYsize = 40.0

# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
fovx = DXsize/ro
fovy = DYsize/ro

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
res = 256
pixels_x = 256
pixels_y = 256

# Distance to the source in parsecs
SourceD = 0.05 * PC
Rout = 1000.0
Rstop = 10000.0

# Frequency observed by the camera in Hz
freq = 2.417989e17#230e9

### Image Generation

This cell performs the **ray tracing and radiative transfer** to build the black hole image:

1. **Camera setup**:  
   The observer’s position is converted into native coordinates (`Xcamera`).  

2. **Scaling**:  
   A `scale_factor` accounts for the physical pixel size on the image plane relative to the source distance.  

3. **Geodesic integration**:  
   - For each pixel `(i, j)`, a geodesic is computed starting from the camera.  
   - The geodesic is traced with a maximum of `maxnstep` integration steps (There is no problem in case it overwhelms it. The array will dynamically increase, but a warning will pop up).  
   - Emission along the geodesic is integrated into the corresponding pixel intensity.  

4. **Final normalization**:  
   After the geodesics are integrated, the image intensity is scaled by `freq³` to ensure proper units, converting from the covariant intensity to the one measured.  

In short, this loop computes the **final intensity map** pixel by pixel by tracing geodesics and integrating emission along each trajectory.


In [3]:
# Find camera in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, bhspin, Rout))

# Scales the intensity of each pixel by the real size of each pixel
scale_factor = CalculateScaleFactor(DXsize, DYsize, pixels_x, pixels_y, SourceD, L_unit)

maxnstep = 15000
# Generate geodesics
println("Utilizing $(Threads.nthreads()) threads for geodesic calculation.")
freq_unitless = freq * HPL/(ME * CL * CL)  # Convert frequency to unitless
Image = zeros(Float64, pixels_x, pixels_y)
for i in 0:(pixels_x - 1)
    println("Processing row $i out of $(pixels_x)")

    Threads.@threads for j in 0:(pixels_y - 1)
        traj = Vector{OfTraj}()
        sizehint!(traj, maxnstep)

        nstep = get_pixel(traj, i, j, Xcamera, maxnstep, fovx, fovy, freq_unitless, pixels_x, pixels_y, bhspin, Rh, Rout, Rstop)

        resize!(traj, length(traj))
        integrate_emission!(traj, length(traj), Image, i + 1, j + 1, freq, bhspin)

    end
end
Image *= freq^3;

Utilizing 4 threads for geodesic calculation.
Processing row 0 out of 40
Processing row 1 out of 40
Processing row 2 out of 40
Processing row 3 out of 40
Processing row 4 out of 40
Processing row 5 out of 40
Processing row 6 out of 40
Processing row 7 out of 40
Processing row 8 out of 40
Processing row 9 out of 40
Processing row 10 out of 40
Processing row 11 out of 40
Processing row 12 out of 40
Processing row 13 out of 40
Processing row 14 out of 40
Processing row 15 out of 40
Processing row 16 out of 40
Processing row 17 out of 40
Processing row 18 out of 40
Processing row 19 out of 40
Processing row 20 out of 40
Processing row 21 out of 40
Processing row 22 out of 40
Processing row 23 out of 40
Processing row 24 out of 40
Processing row 25 out of 40
Processing row 26 out of 40
Processing row 27 out of 40
Processing row 28 out of 40
Processing row 29 out of 40
Processing row 30 out of 40
Processing row 31 out of 40
Processing row 32 out of 40
Processing row 33 out of 40
Processing r

### Diagnostics from `OutputStokesParameters`

After generating the image, `OutputStokesParameters` prints basic diagnostics about the intensity map:

- **Scale**: normalization factor applied to convert pixel values into physical units.  
- **imax, jmax**: pixel indices `(i, j)` where the maximum intensity `Imax` is located.  
- **Imax**: maximum specific intensity in the image (at the brightest pixel).  
- **Iavg**: average specific intensity across all pixels.  
- **freq_cgs**: observing frequency in cgs units (Hz).  
- **Ftot**: total flux density (in cgs units, e.g. erg s⁻¹ cm⁻² Hz⁻¹) obtained by summing all pixel intensities.  
- **νLν (nuLnu)**: luminosity at frequency ν, computed as `ν × Lν`, where `Lν` is the spectral luminosity corresponding to `Ftot`.


In [4]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 3.516934446296018e+01
imax = 15, jmax = 20, Imax = 9.670671453216482e-7, Iavg = 4.5977790459928304e-7
Using freq_cgs = 2.3e11, Ftot = 0.025872140005296272
nuLnu = 4.307320010024429e32


Plotting the final image with the appropriate angular spacing 

In [ ]:
using CairoMakie
using Printf

d_kpc = 7.78
d_cm = d_kpc * 3.086e21           # distance in cm
fov_rg = 30                       # field of view in gravitational radii
half_fov_rg = fov_rg / 2

# Angular resolution
theta_rad = (half_fov_rg * L_unit) / d_cm   # half FOV in radians
theta_μas = theta_rad * MUAS_PER_RAD        # convert to μas
xlims = (-theta_μas, theta_μas)
ylims = (-theta_μas, theta_μas)

# Generate mock image (or load your 128x128 array here)
img = Image

# Axes
Ny, Nx = size(img)
x = range(xlims[1], xlims[2], length=Nx)
y = range(ylims[1], ylims[2], length=Ny)

# Plot
fig = Figure(size = (600, 500))
ax = Axis(fig[1, 1],
    xlabel = "Relative R.A [μas]",
    ylabel = "Relative Dec [μas]",
    xlabelsize=24,
    ylabelsize=24,
    xticklabelsize=20,
    yticklabelsize=20,
    limits = (xlims, ylims)
)

# Heatmap with color range
crange = extrema(img)
hm = heatmap!(ax, x, y, img; colormap=:hot, colorrange=crange)

# Colorbar
Colorbar(fig[1, 2], hm;
    label = "Intensity",
    labelsize = 20,
    ticklabelsize = 16,
    width = 15
)

fig
